In [1]:
import pandas as pd
import re
import requests
from io import BytesIO
import numpy as np
from typing import Optional

In [2]:
import pandas as pd

file_id = "1zdZJxidp0DHzp0-MsvEMD7lxP3xsBwA3J4P9BAIvn0c"
sheet_name = "HIV2015"

url = (
    f"https://docs.google.com/spreadsheets/d/{file_id}/export"
    f"?format=xlsx&sheet={sheet_name}"
)

main_df = pd.read_excel(url, sheet_name=sheet_name)

main_df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,1,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 74,Unnamed: 75,Unnamed: 76,Unnamed: 77,Unnamed: 78,"Used in Col L, N",Unnamed: 80,Unnamed: 81,Unnamed: 82,Unnamed: 83
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,OVERALL,OVERALL,OVERALL,OVERALL
1,Country,WHO Region,Population,Geographical Region,WHO Group,DALY,Adult DALYs,Children DALYs,Retention Rate,Retention Rate (ADULT),...,All ages,Children (0-14),Adults (15+),Year,NaN,http://apps.who.int/gho/data/node.main.626?lan...,Estimated antiretroviral therapy coverage amon...,Reported number of people receiving antiretrov...,Cleaned coverage,Cleaned number of people receiving antiretrovi...
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,max of RR and 97.14 (1/35),max of RR and 97.14 (1/35),...,72,60,73,2015,NaN,Country,2015,2015,NaN,NaN
3,Afghanistan,EMR,33736494,"East, South and South-East Asia",A,10752.548,9224.366,1528.182,72,73,...,92,100,92,2015,NaN,Afghanistan,5 [3-12],364,0.05,364
4,Albania,EUR,2880703,Europe and Central Asia,A,98.495261,96.59686,1.898401,92,92,...,92,77,92,2015,NaN,Albania,No data,423,NaN,423


In [3]:
new_header = main_df.iloc[1]
main_df = main_df.iloc[3:]
main_df.columns = new_header
main_df = main_df.reset_index(drop=True)

main_df.head()

1,Country,WHO Region,Population,Geographical Region,WHO Group,DALY,Adult DALYs,Children DALYs,Retention Rate,Retention Rate (ADULT),...,All ages,Children (0-14),Adults (15+),Year,NaN,http://apps.who.int/gho/data/node.main.626?lang=en,Estimated antiretroviral therapy coverage among people living with HIV (%),Reported number of people receiving antiretroviral therapy,Cleaned coverage,Cleaned number of people receiving antiretroviral therapy
0,Afghanistan,EMR,33736494,"East, South and South-East Asia",A,10752.548,9224.366,1528.182,72,73,...,92,100,92,2015,NaN,Afghanistan,5 [3-12],364,0.05,364
1,Albania,EUR,2880703,Europe and Central Asia,A,98.495261,96.59686,1.898401,92,92,...,92,77,92,2015,NaN,Albania,No data,423,NaN,423
2,Algeria,AFR,39871528,Middle East and North Africa,A,11586.0447,11055.12,530.9247,92,92,...,100,NaN,NaN,2015,NaN,Algeria,90 [70->95],7 915,0.9,7915
3,American Samoa,WPR,55537,NaN,A,28.518669,25.84525,2.673419,97.14,97.14,...,66,NaN,NaN,2015,NaN,Andorra,High-income country,No data,NaN,NaN
4,Andorra,EUR,78014,NaN,A,83.36974,83.20017,0.16957,97.14,97.14,...,85,100,85,2015,NaN,Angola,29 [20-40],90 204,0.29,90204


In [4]:
# ── Country input columns (0-based, matching Excel letters) ──
COL_G  = 6    # Adult DALYs
COL_H  = 7    # Children DALYs
COL_I  = 8    # Retention Rate (%)
COL_Q  = 16   # Adult % Treatment Coverage
COL_T  = 19   # Children % Treatment Coverage

# ── Impact score columns (actual Excel values, for comparison) ──
COL_Y  = 24   # Y  = 3TC  (first drug)
COL_AI = 35   # AJ = Overall Treatment Impact (excluded)

# ── Regimen lookup columns ──
COL_AS = 44   # Regimen name string
COL_AX = 49   # Number of drugs in regimen
COL_AY = 50   # adult_x = adult_proportion × adult_efficacy
COL_AZ = 51   # child_x = child_proportion × child_efficacy

# ── Regimen rows used by the Excel formula (0-based raw indices) ──
# Excel 1-based rows: 5–9, 12–18, 21–29, 31–37
REGIMEN_ROWS = (
    list(range(4,  9))   # Excel  5– 9  first-line adult
  + list(range(11, 18))  # Excel 12–18  first-line children
  + list(range(20, 29))  # Excel 21–29  second-line
  + list(range(30, 37))  # Excel 31–37  additional second-line
)

In [5]:
def to_float(val, default=0.0):
    """Parse a cell to float, stripping commas and % signs."""
    if pd.isna(val):
        return default
    try:
        return float(str(val).replace(",", "").replace("%", "").strip())
    except (ValueError, TypeError):
        return default

def is_numeric(val):
    if pd.isna(val): return False
    try:   float(str(val).replace(",", "")); return True
    except: return False

In [6]:
import pandas as pd
import numpy as np

In [7]:
raw = pd.read_excel(url, sheet_name=sheet_name, header=None)

In [8]:
regimens = []
for r in REGIMEN_ROWS:
    regimens.append({
        "regimen": str(raw.iloc[r, COL_AS]).strip(),
        "n_drugs": to_float(raw.iloc[r, COL_AX], np.nan),
        "adult_x": to_float(raw.iloc[r, COL_AY]),
        "child_x": to_float(raw.iloc[r, COL_AZ]),
    })

regimens_df = pd.DataFrame(regimens)
regimens_df

,regimen,n_drugs,adult_x,child_x
0,TDF + 3TC + EFV,3.0,0.229558,0.000000
1,TDF + FTC + EFV,3.0,0.156963,0.000000
2,AZT + 3TC + NVP,3.0,0.149593,0.314704
3,TDF + 3TC + NVP,3.0,0.089245,0.000000
4,AZT + 3TC + EFV,3.0,0.056363,0.050498
5,ABC + 3TC + LPV/r,3.0,0.000000,0.078785
6,ABC + 3TC + EFV,3.0,0.000000,0.084746
7,ABC + 3TC + NVP,3.0,0.000000,0.031264
8,d4T + 3TC + NVP,3.0,0.000000,0.022590
9,d4T + 3TC + LPV/r,3.0,0.000000,0.008932


In [9]:
# Drug names from main_df column labels
drug_cols  = [c for c in main_df.columns[COL_Y:COL_AI]
              if pd.notna(c) and str(c).strip() not in ("", "nan")]
drug_names = [str(d).strip() for d in drug_cols]

# Store actual Excel scores separately for comparison
actual_scores = main_df[drug_cols].copy()
actual_scores.columns = [f"actual_{d}" for d in drug_names]
for col in actual_scores.columns:
    actual_scores[col] = actual_scores[col].apply(to_float)

# Clean input df — inputs only, no impact scores
df = pd.DataFrame({
    "Country":       main_df.iloc[:, 0].values,
    "WHO Region":    main_df.iloc[:, 1].values,
    "adult_daly":    main_df.iloc[:, COL_G].apply(to_float).values,
    "child_daly":    main_df.iloc[:, COL_H].apply(to_float).values,
    "retention_pct": main_df.iloc[:, COL_I].apply(to_float).values,
    # read_excel already parses % cells as fractions (e.g. 4.99% → 0.0499)
    # so store as-is — do NOT multiply by 100
    "adult_cov":     pd.to_numeric(main_df.iloc[:, COL_Q], errors="coerce").fillna(0).values,
    "child_cov":     pd.to_numeric(main_df.iloc[:, COL_T], errors="coerce").fillna(0).values,
})
df = pd.concat([df, actual_scores.reset_index(drop=True)], axis=1)

# Keep valid country rows only
df = df[df["adult_daly"] > 0].reset_index(drop=True)

print(f"Countries: {len(df)}  |  Drugs: {drug_names}")
print(f"Afghanistan adult_cov = {df.loc[0,'adult_cov']:.4f}  (should be ~0.0499)")
df.head()

Countries: 180  |  Drugs: ['3TC', 'ABC', 'AZT', 'ddl', 'd4T', 'EFV', 'FTC', 'LPV/r', 'NVP', 'TDF', 'ATV/r']
Afghanistan adult_cov = 0.0499  (should be ~0.0499)


,Country,WHO Region,adult_daly,child_daly,retention_pct,adult_cov,child_cov,actual_3TC,actual_ABC,actual_AZT,actual_ddl,actual_d4T,actual_EFV,actual_FTC,actual_LPV/r,actual_NVP,actual_TDF,actual_ATV/r
0,Afghanistan,EMR,9224.36600,1528.182000,72.00,0.049853,0.11,35.667802,3.483114,15.831563,0.102642,0.523836,21.363380,7.057698,4.612259,16.425901,23.749159,0.412628
1,Albania,EUR,96.59686,1.898401,92.00,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Algeria,AFR,11055.12000,530.924700,92.00,0.884086,0.95,186.166832,3.714239,70.801802,0.088267,0.455925,140.014900,49.132222,15.754865,77.092066,164.992819,2.513475
3,American Samoa,WPR,25.84525,2.673419,97.14,0.363716,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Andorra,EUR,83.20017,0.169570,97.14,0.363716,0.00,0.176927,0.000706,0.065802,0.000000,0.000000,0.136883,0.049826,0.014843,0.072249,0.165287,0.002777


In [10]:
def calc_impact_score(row, drug):
    """
    Replicates the Excel formula for one (country, drug) pair.
    Returns 0.0 on any error, matching Excel IFERROR behaviour.

    Note: adult_cov and child_cov are already fractions (read_excel
    converts % cells automatically), so no /100 needed here.
    """
    G = row["adult_daly"]
    H = row["child_daly"]
    I = row["retention_pct"]   # percent e.g. 72.0
    Q = row["adult_cov"]       # already a fraction e.g. 0.0499
    T = row["child_cov"]       # already a fraction e.g. 0.11

    if I >= 100:
        return 0.0
    retention_factor = 100.0 / (100.0 - I)

    total = 0.0
    for _, reg in regimens_df.iterrows():
        if drug.lower() not in reg["regimen"].lower():
            continue

        AX, AY, AZ = reg["n_drugs"], reg["adult_x"], reg["child_x"]

        if pd.isna(AX) or AX == 0:
            continue

        if AY > 0:
            d = 1.0 - Q * AY
            if d != 0: total += G * Q * AY / d / AX

        if AZ > 0:
            d = 1.0 - T * AZ
            if d != 0: total += H * T * AZ / d / AX

    try:
        result = total / retention_factor
        return result if np.isfinite(result) else 0.0
    except ZeroDivisionError:
        return 0.0

In [11]:
for drug in drug_names:
    df[f"calc_{drug}"] = df.apply(calc_impact_score, drug=drug, axis=1)

print("Done.")
df[[c for c in df.columns if c.startswith("calc_")]].head()

Done.


,calc_3TC,calc_ABC,calc_AZT,calc_ddl,calc_d4T,calc_EFV,calc_FTC,calc_LPV/r,calc_NVP,calc_TDF,calc_ATV/r
0,35.667802,3.483114,15.831563,0.102642,0.523836,21.363380,7.057698,4.612259,16.425901,23.046045,0.412628
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,186.166832,3.714239,70.801802,0.088267,0.455925,140.014900,49.132222,15.754865,77.092066,160.703635,2.513475
3,0.054960,0.000219,0.020441,0.000000,0.000000,0.042521,0.015478,0.004611,0.022444,0.049874,0.000863
4,0.176927,0.000706,0.065802,0.000000,0.000000,0.136883,0.049826,0.014843,0.072249,0.160553,0.002777


In [12]:
print(f"{'Drug':<8} {'Max|Diff|':>10} {'Mean|Diff|':>12} {'<0.01(%)':>10} {'<0.1(%)':>9}")
print("-" * 54)
for d in drug_names:
    diff = (df[f"calc_{d}"] - df[f"actual_{d}"]).abs()
    print(f"{d:<8} {diff.max():>10.4f} {diff.mean():>12.4f}"
          f" {(diff < 0.01).mean()*100:>9.1f}% {(diff < 0.1).mean()*100:>8.1f}%")

Drug      Max|Diff|   Mean|Diff|   <0.01(%)   <0.1(%)
------------------------------------------------------
3TC          0.0550       0.0003      99.4%    100.0%
ABC          0.0002       0.0000     100.0%    100.0%
AZT          0.0204       0.0001      99.4%    100.0%
ddl          0.0000       0.0000     100.0%    100.0%
d4T          0.0000       0.0000     100.0%    100.0%
EFV          0.0425       0.0002      99.4%    100.0%
FTC          0.0155       0.0001      99.4%    100.0%
LPV/r        0.0046       0.0000     100.0%    100.0%
NVP          0.0224       0.0001      99.4%    100.0%
TDF       2710.0269      75.6455      26.7%     34.4%
ATV/r        0.0009       0.0000     100.0%    100.0%


In [15]:
df.columns

Index(['Country', 'WHO Region', 'adult_daly', 'child_daly', 'retention_pct',
       'adult_cov', 'child_cov', 'actual_3TC', 'actual_ABC', 'actual_AZT',
       'actual_ddl', 'actual_d4T', 'actual_EFV', 'actual_FTC', 'actual_LPV/r',
       'actual_NVP', 'actual_TDF', 'actual_ATV/r', 'calc_3TC', 'calc_ABC',
       'calc_AZT', 'calc_ddl', 'calc_d4T', 'calc_EFV', 'calc_FTC',
       'calc_LPV/r', 'calc_NVP', 'calc_TDF', 'calc_ATV/r'],
      dtype='object')

In [16]:
df.to_csv("impact_scores.csv", index=False)
print("File saved as csv")

File saved as csv
